# MPPI Solvers — Class Notebook

In this notebook we implement and compare three sampling-based trajectory
optimizers for Model Predictive Control (MPC), all built on top of `evosax`'s
`DistributionBasedAlgorithm` base class.

## The problem

We want to find a sequence of controls $\mathbf{U} = (u_0, u_1, \dots, u_{T-1})$,
$u_t \in \mathbb{R}^{n_u}$, that minimises a cost $J(\mathbf{U})$ obtained by
simulating a dynamical system forward:

$$\mathbf{U}^* = \arg\min_{\mathbf{U}} \; J(\mathbf{U})$$

We flatten $\mathbf{U}$ into a single vector $x \in \mathbb{R}^{T \cdot n_u}$
and treat this as a black-box optimisation problem: we **cannot** differentiate
through the simulator, so we must rely on sampling.

## The ask–tell interface

All solvers share the same loop structure provided by `evosax`:

```
state = es.init(key, x_init, params)          # initialise distribution

for generation in range(N):
    population, state = es.ask(key, state, params)   # sample K candidates
    fitness = evaluate(population)                    # simulate each candidate
    state, metrics = es.tell(key, population,        # update distribution
                             fitness, state, params)
```

| Step | What it does |
|------|-------------|
| **`ask`** | Draws $K$ candidate solutions $\{x^{(k)}\}_{k=1}^K$ from the current distribution $p_\theta$. Returns the population matrix of shape $(K, T \cdot n_u)$. |
| **evaluate** | Runs the simulator on all $K$ candidates in parallel and returns a fitness vector $c \in \mathbb{R}^K$ (lower = better). This is the only part you do **not** implement. |
| **`tell`** | Receives the fitness scores and updates the distribution parameters $\theta$ (mean, covariance, …). This is where the algorithm logic lives. |

You will implement `_ask` and `_tell` for three algorithms:

1. **MPPI** — weights all $K$ samples by an exponential of their cost.
2. **MPPIElite** — same, but restricts the update to the best $K_\text{elite}$ samples.
3. **MPPICMA** — additionally adapts a full covariance matrix so the sampling
   distribution learns the correlation structure of good trajectories.

## Step 1 — Solver definitions

## Warm-up — Running a solver end-to-end with SimpleES

Before implementing your own solvers, let's run a complete ask-tell loop using `SimpleES` from `evosax` on the CartPole task. This shows you the full pipeline you will be working with:

1. Instantiate a solver and a task
2. Initialise the solver state
3. Loop: `ask` → evaluate → `tell`
4. Inspect the best solution found

Read through this cell carefully — the structure is exactly what you will reuse for your own solvers.

In [ ]:
import os

os.environ["MUJOCO_GL"] = "egl"

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

from evosax.algorithms import SimpleES
from cartpole_task import CartpoleSwingUp

# --- Task setup ---
N_ENVS = 256
task = CartpoleSwingUp(N_ENVS)
control_init = task.get_warm_start()  # zero control trajectory, shape (T, nu)

print(f"Control horizon : T={task.N_STEPS} steps, nu={task.mj_model.nu} actuator")
print(f"Search dimension: {control_init.size}  (T × nu flattened by evosax)")

# --- Solver setup ---
es = SimpleES(population_size=N_ENVS, solution=control_init)
params = es.default_params.replace(std_init=0.25)

key = jax.random.key(0)
state = es.init(key, control_init, params)

# --- Ask-tell loop ---
NUM_GENERATIONS = 100
best_fitness_log = []

for _ in tqdm(range(NUM_GENERATIONS), desc="SimpleES"):
    key, key_ask, key_tell = jax.random.split(key, 3)

    # 1. Sample a population of candidate control trajectories
    population, state = es.ask(key_ask, state, params)
    # population shape: (N_ENVS, T*nu)

    # 2. Evaluate each candidate by rolling it out in the simulator
    fitness = jnp.array(task.cost_function(np.array(population)))
    # fitness shape: (N_ENVS,)  — lower is better

    # 3. Update the solver distribution based on fitness scores
    state, metrics = es.tell(key_tell, population, fitness, state, params)

    best_fitness_log.append(float(metrics["best_fitness"]))

print(f"\nFinal best fitness: {best_fitness_log[-1]:.4f}")

# --- Plot convergence ---
plt.figure(figsize=(7, 3))
plt.plot(best_fitness_log, linewidth=2)
plt.xlabel("Generation")
plt.ylabel("Best fitness")
plt.title("SimpleES on CartPole Swing-Up")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import mujoco
import imageio
from IPython.display import Video, display

os.makedirs("Figures", exist_ok=True)

# Roll out the best control trajectory found by SimpleES
best_control = np.array(state.best_solution.reshape((task.N_STEPS, -1)))

data = mujoco.MjData(task.mj_model)
data.qpos[:] = task.q_init.copy()
mujoco.mj_forward(task.mj_model, data)

camera_id = mujoco.mj_name2id(task.mj_model, mujoco.mjtObj.mjOBJ_CAMERA, "fixed")
fps = int(round(1.0 / task.dt))
video_path = "Figures/warmup_simplees_cartpole.mp4"

with mujoco.Renderer(task.mj_model, width=640, height=480) as renderer:
    with imageio.get_writer(video_path, fps=fps) as writer:
        for t in range(task.N_STEPS):
            for _ in range(task.N_HOLD):
                data.ctrl[:] = best_control[t]
                mujoco.mj_step(task.mj_model, data)
                renderer.update_scene(data, camera=camera_id)
                writer.append_data(renderer.render())

print(f"Saved to {video_path}")
display(Video(video_path, width=640, height=480))

### Shared imports and base classes

All three solvers subclass `DistributionBasedAlgorithm`, which handles
bookkeeping (best solution tracking, generation counter) and exposes the
public `ask` / `tell` API. You only need to implement four private methods:

| Method | Signature | What to fill in |
|--------|-----------|-----------------|
| `_init` | `(key, params) → State` | Initialise the distribution parameters (mean, std, …) |
| `_ask` | `(key, state, params) → (population, state)` | Sample $K$ candidates from the current distribution |
| `_tell` | `(key, population, fitness, state, params) → state` | Update the distribution given the fitness scores |

The **`State`** dataclass holds everything that changes across generations
(mean, covariance, …). The **`Params`** dataclass holds fixed hyperparameters
(temperature, std, …). Both are immutable JAX pytrees — use `.replace(...)` to
return an updated copy.

In [ ]:
from collections.abc import Callable

import jax
import jax.numpy as jnp
from flax import struct
import mujoco

from evosax.core.fitness_shaping import identity_fitness_shaping_fn
from evosax.types import Fitness, Population, Solution
from evosax.algorithms.distribution_based.base import (
    DistributionBasedAlgorithm,
    Params as BaseParams,
    State as BaseState,
    metrics_fn,
)

print("✓ Base imports ready")

### 1. MPPI

**Idea.** Maintain a Gaussian distribution $\mathcal{N}(\mu, \sigma^2 I)$ over
the flattened control trajectory. Each generation:

1. **Ask** — draw $K$ candidates by adding independent Gaussian noise to the mean:
$$x^{(k)} = \mu + \sigma\,\varepsilon^{(k)}, \qquad \varepsilon^{(k)} \sim \mathcal{N}(0, I)$$

2. **Evaluate** — simulate all $K$ candidates and collect costs $c^{(k)} = J(x^{(k)})$.

3. **Tell** — move the mean toward the low-cost candidates using importance weights:
$$w_k = \frac{\exp\!\left(-\dfrac{c_k - \beta}{\lambda\,\hat\sigma_c}\right)}{\displaystyle\sum_j \exp\!\left(-\dfrac{c_j - \beta}{\lambda\,\hat\sigma_c}\right)}, \qquad \mu \leftarrow \sum_{k=1}^K w_k\, x^{(k)}$$

where $\beta = \min_k c_k$ shifts the exponent for numerical stability,
$\hat\sigma_c = \text{std}(\mathbf{c})$ normalises by the cost spread, and
$\lambda > 0$ is the **temperature** hyperparameter:
- small $\lambda$ → winner-takes-all (only the best sample matters)
- large $\lambda$ → uniform averaging (all samples contribute equally)

The standard deviation $\sigma$ is **fixed** and does not adapt.

**Your task:** implement `_ask` (sample the population) and `_tell` (compute weights and update $\mu$).

In [ ]:
@struct.dataclass
class MPPIState(BaseState):
    mean: jax.Array
    std: jax.Array


@struct.dataclass
class MPPIParams(BaseParams):
    std_init: float
    temperature: float


class MPPI(DistributionBasedAlgorithm):
    """Model Predictive Path Integral solver.

    Pure MPPI that weights all samples using importance weighting.
    Uses the entire population for the distribution update.
    """

    def __init__(
        self,
        population_size: int,
        solution: Solution,
        fitness_shaping_fn: Callable = identity_fitness_shaping_fn,
        metrics_fn: Callable = metrics_fn,
    ):
        super().__init__(population_size, solution, fitness_shaping_fn, metrics_fn)

    @property
    def _default_params(self) -> MPPIParams:
        return MPPIParams(
            std_init=0.25,
            temperature=1.0,
        )

    def _init(self, key: jax.Array, params: MPPIParams) -> MPPIState:
        return MPPIState(
            mean=jnp.zeros(self.num_dims),
            std=jnp.array(params.std_init),
            best_solution=jnp.full((self.num_dims,), jnp.nan),
            best_fitness=jnp.inf,
            generation_counter=0,
        )

    def _ask(
        self,
        key: jax.Array,
        state: MPPIState,
        params: MPPIParams,
    ) -> tuple[Population, MPPIState]:
        # TODO: sample self.population_size candidates from N(mean, std^2 * I)
        #   Hint: use jax.random.normal to draw a (population_size, num_dims) noise matrix
        #   then shift and scale by state.mean and state.std
        raise NotImplementedError

    def _tell(
        self,
        key: jax.Array,
        population: Population,
        fitness: Fitness,
        state: MPPIState,
        params: MPPIParams,
    ) -> MPPIState:
        # TODO: compute importance weights from fitness scores
        #   Step 1: shift by beta = min(fitness) for numerical stability
        #   Step 2: normalise the cost spread by std(fitness) + 1e-8
        #   Step 3: compute weights as exp(-(fitness - beta) / (temperature * cost_scale))
        #   Step 4: normalise weights to sum to 1
        #   Step 5: update mean as the weighted average of the population: weights @ population
        raise NotImplementedError


print("✓ MPPI defined")

### 2. MPPIElite

**Idea.** Identical sampling to MPPI, but the mean update uses only the
$K_\text{elite} = \lfloor\rho \cdot K\rfloor$ **lowest-cost** samples
(with $\rho$ the `elite_ratio`, e.g. 10 %). The importance weights are
recomputed over this elite subset:

$$\mathcal{E} = \text{argsort}(\mathbf{c})[{:K_\text{elite}}]$$

$$w_k = \frac{\exp\!\left(-\dfrac{c_k - c_{\min}}{\lambda\,\hat\sigma_\mathcal{E}}\right)}{\displaystyle\sum_{j \in \mathcal{E}} \exp\!\left(-\dfrac{c_j - c_{\min}}{\lambda\,\hat\sigma_\mathcal{E}}\right)}, \quad k \in \mathcal{E}$$

$$\mu \leftarrow \sum_{k \in \mathcal{E}} w_k\, x^{(k)}$$

**Why does this help?** MPPI can waste weight on mediocre samples if the cost
distribution has a long tail. Restricting to the elite set makes the update
robust to outlier trajectories and often converges faster.

**Your task:** implement `_tell` — sort by fitness, slice the elite subset,
recompute weights, and update $\mu$. The `_ask` method is identical to MPPI.

In [ ]:
@struct.dataclass
class MPPIEliteState(BaseState):
    mean: jax.Array
    std: jax.Array


@struct.dataclass
class MPPIEliteParams(BaseParams):
    std_init: float
    temperature: float
    elite_ratio: float


class MPPIElite(DistributionBasedAlgorithm):
    """Model Predictive Path Integral with elite set selection.

    Extends MPPI by using only the best elite_ratio fraction of samples
    for the mean update.
    """

    def __init__(
        self,
        population_size: int,
        solution: Solution,
        elite_ratio: float = 0.1,
        fitness_shaping_fn: Callable = identity_fitness_shaping_fn,
        metrics_fn: Callable = metrics_fn,
    ):
        super().__init__(population_size, solution, fitness_shaping_fn, metrics_fn)
        self._k_elite = max(1, int(elite_ratio * population_size))

    @property
    def _default_params(self) -> MPPIEliteParams:
        return MPPIEliteParams(
            std_init=0.25,
            temperature=1.0,
            elite_ratio=0.1,
        )

    def _init(self, key: jax.Array, params: MPPIEliteParams) -> MPPIEliteState:
        return MPPIEliteState(
            mean=jnp.zeros(self.num_dims),
            std=jnp.array(params.std_init),
            best_solution=jnp.full((self.num_dims,), jnp.nan),
            best_fitness=jnp.inf,
            generation_counter=0,
        )

    def _ask(
        self,
        key: jax.Array,
        state: MPPIEliteState,
        params: MPPIEliteParams,
    ) -> tuple[Population, MPPIEliteState]:
        # TODO: identical to MPPI._ask — sample population_size candidates from N(mean, std^2 * I)
        #   Hint: copy your MPPI _ask implementation here
        raise NotImplementedError

    def _tell(
        self,
        key: jax.Array,
        population: Population,
        fitness: Fitness,
        state: MPPIEliteState,
        params: MPPIEliteParams,
    ) -> MPPIEliteState:
        # TODO: restrict the mean update to the best self._k_elite samples
        #   Step 1: sort the population by fitness using jnp.argsort(fitness)
        #   Step 2: slice the top-k elite samples and their fitness scores
        #   Step 3: recompute importance weights over the elite subset only
        #           (same formula as MPPI: shift by min, normalise by std, exponentiate)
        #   Step 4: normalise weights to sum to 1
        #   Step 5: update mean as the weighted average of the elite population
        raise NotImplementedError


print("✓ MPPIElite defined")

### 3. MPPICMA

**Idea.** Replace the isotropic Gaussian $\mathcal{N}(\mu, \sigma^2 I)$ used in
MPPI with a **full-covariance** Gaussian $\mathcal{N}(\mu, \Sigma)$. This lets
the distribution learn the **correlation structure** of good trajectories — e.g.
if a large hip angle at $t=0$ should always be paired with a large knee angle at
$t=1$, the off-diagonal entries of $\Sigma$ will capture that.

**Ask** — sampling from $\mathcal{N}(\mu, \Sigma)$ requires factorising $\Sigma$.
We use the **eigen decomposition** $\Sigma = B\,\text{diag}(D^2)\,B^\top$
(where $B$ are the eigenvectors and $D$ the square roots of the eigenvalues),
then sample as:

$$x^{(k)} = \mu + B\,\text{diag}(D)\,\varepsilon^{(k)}, \qquad \varepsilon^{(k)} \sim \mathcal{N}(0, I)$$

This works because $\text{Cov}(B\,\text{diag}(D)\,\varepsilon) = B\,\text{diag}(D^2)\,B^\top = \Sigma$.
In code this is written as `(noise @ diag(D)) @ B.T` for a batch of noise vectors.

**Tell** — the weights are computed exactly as in MPPI (exponential of shifted,
normalised costs):

$$w_k = \frac{\exp\!\left(-\dfrac{c_k - \beta}{\lambda\,\hat\sigma_c}\right)}{\displaystyle\sum_j \exp\!\left(-\dfrac{c_j - \beta}{\lambda\,\hat\sigma_c}\right)}$$

These weights are then used to update both the mean and the covariance with
exponential moving averages:

$$\mu \;\leftarrow\; (1 - \alpha_\mu)\,\mu + \alpha_\mu \sum_k w_k\, x^{(k)}$$

$$\Sigma \;\leftarrow\; (1 - \alpha_\Sigma)\,\Sigma
    + \alpha_\Sigma \underbrace{\sum_k w_k\,(x^{(k)} - \mu)(x^{(k)} - \mu)^\top}_{\text{weighted sample covariance}}$$

where $\alpha_\mu$ (`lr_mean`) and $\alpha_\Sigma$ (`lr_cov`) are learning
rates. The deviations are computed from the **old** $\mu$ (before the mean
update) to match Algorithm 6 in the paper.

**Your task:** implement `_ask` (eigen decomposition sampling) and `_tell`
(same exponential weights as MPPI, weighted covariance update, EMA for both
$\mu$ and $\Sigma$).

In [ ]:
@struct.dataclass
class MPPICMAState(BaseState):
    mean: jax.Array  # Shape: (T*nu,) - flattened mean control trajectory
    cov: jax.Array  # Shape: (T*nu, T*nu) - full covariance matrix


@struct.dataclass
class MPPICMAParams(BaseParams):
    std_init: float
    temperature: float
    lr_mean: float  # Learning rate for mean
    lr_cov: float  # Learning rate for covariance


class MPPICMA(DistributionBasedAlgorithm):
    """Model Predictive Path Integral with Covariance Matrix Adaptation.

    Uses a full (T*nu) x (T*nu) covariance matrix, capturing cross-timestep
    correlations. More expensive, but more expressive.
    """

    def __init__(
        self,
        population_size: int,
        solution: Solution,
        fitness_shaping_fn: Callable = identity_fitness_shaping_fn,
        metrics_fn: Callable = metrics_fn,
    ):
        super().__init__(population_size, solution, fitness_shaping_fn, metrics_fn)

    @property
    def _default_params(self) -> MPPICMAParams:
        return MPPICMAParams(
            std_init=0.25,
            temperature=1.0,
            lr_mean=1.0,
            lr_cov=0.1,
        )

    def _init(self, key: jax.Array, params: MPPICMAParams) -> MPPICMAState:
        cov = jnp.eye(self.num_dims) * (params.std_init**2)
        return MPPICMAState(
            mean=jnp.zeros(self.num_dims),
            cov=cov,
            best_solution=jnp.full((self.num_dims,), jnp.nan),
            best_fitness=jnp.inf,
            generation_counter=0,
        )

    def _ask(
        self,
        key: jax.Array,
        state: MPPICMAState,
        params: MPPICMAParams,
    ) -> tuple[Population, MPPICMAState]:
        # TODO: sample from N(mean, cov) using the eigen decomposition of the covariance matrix
        #   Step 1: decompose cov via eigen_decomposition(state.cov) → returns (_, B, D)
        #           where B are the eigenvectors (shape: num_dims x num_dims)
        #           and D are the square roots of the eigenvalues (shape: num_dims,)
        #   Step 2: draw standard normal noise of shape (population_size, num_dims)
        #   Step 3: transform noise to match the covariance: (noise @ jnp.diag(D)) @ B.T
        #           this is equivalent to sampling x = mean + B @ diag(D) @ eps
        raise NotImplementedError

    def _tell(
        self,
        key: jax.Array,
        population: Population,
        fitness: Fitness,
        state: MPPICMAState,
        params: MPPICMAParams,
    ) -> MPPICMAState:
        # TODO: update both the mean and the covariance using importance-weighted samples
        #   Step 1: compute importance weights (same as MPPI: shift, normalise, exponentiate, sum to 1)
        #   Step 2: compute deviations from the OLD mean (before updating it):
        #           deviations = population - state.mean  shape: (population_size, num_dims)
        #   Step 3: compute the weighted sample covariance:
        #           sum_k w_k * outer(deviation_k, deviation_k)
        #           Hint: use jnp.einsum("ij,ik->ijk", deviations, deviations) for the outer products
        #           then sum with weights[:, None, None] * outer, add 1e-8 * I for regularisation
        #   Step 4: update covariance with EMA: (1 - lr_cov) * cov + lr_cov * cov_update
        #   Step 5: update mean with EMA: (1 - lr_mean) * mean + lr_mean * (weights @ population)
        #   Return state.replace(mean=mean, cov=cov)
        raise NotImplementedError


print("✓ MPPICMA defined")

## Step 2 — Environment 1: CartPole Swing-Up

### System description

The CartPole is a classic control benchmark: a cart that slides along a rail with a pole attached at a hinge on top.

```
         o   ← pole tip
         |
         |   pole (rigid, length ~0.6 m)
         |
    [===cart===]  ← slides left/right
  ──────────────────  rail
```

The cart is actuated by a horizontal force applied directly to it. The pole is **unactuated** — it moves only through the reaction forces from the cart. This makes the problem non-trivially underactuated.

### State space

| Variable | Symbol | Description |
|----------|--------|-------------|
| Cart position | $x$ | Horizontal position of the cart [m] |
| Pole angle | $\theta$ | Angle of the pole from vertical [rad] |
| Cart velocity | $\dot{x}$ | [m/s] |
| Pole angular velocity | $\dot{\theta}$ | [rad/s] |

The **initial state** is the pole hanging straight down: $q_0 = (x, \theta) = (0, \pi)$, velocities zero.  
The **goal** is the pole balanced upright: $q^* = (0, 0)$, velocities zero.

### Action space

| Variable | Range | Description |
|----------|-------|-------------|
| Force $u$ | $[-2, 2]$ N | Horizontal force on the cart |

The control is applied with **zero-order hold**: each of the $T=20$ control steps is held for $N_\text{hold}=4$ simulator steps (timestep $\Delta t = $ `task.dt`).

### Cost function

The cost penalises deviation from the goal at each step plus a large terminal cost:

$$J = \sum_{t=0}^{T-1} \left[ 100\,\max(|x_t| - 0.95, 0)^2 + 10^{-4}\,u_t^2 + 100\,\max(|u_t| - 2, 0)^2 \right] + J_\text{terminal}$$

$$J_\text{terminal} = 10\,x_T^2 + 10\,\theta_T^2 + 10\,(\dot{x}_T^2 + \dot{\theta}_T^2)$$

The running cost enforces the rail boundary ($|x| \le 0.95$ m) and penalises large controls. The terminal cost drives the system to the upright balanced state.

### What makes this hard?

- The system must first **swing the pole up** (non-minimum-phase behaviour) before it can balance it — there is no direct path to the goal.
- The cart is **rail-bounded**: aggressive swinging strategies risk hitting the wall.
- With only $T=20$ steps, the solver has a short horizon to find the right swing trajectory.

In [ ]:
import os

os.environ["MUJOCO_GL"] = "egl"

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from cartpole_task import CartpoleSwingUp

N_ENVS = 256
task = CartpoleSwingUp(N_ENVS)
control_init = task.get_warm_start()

print("Task       : CartPole Swing-Up")
print(f"Steps      : {task.N_STEPS}")
print(f"Control dim: {control_init.shape}  (T × nu)")
print(f"Population : {N_ENVS}")

## Step 3 — Run and compare all three solvers

We run each solver for 200 generations on the same task and compare their
convergence curves.


In [ ]:
import jax
import jax.numpy as jnp
from evosax.algorithms import CMA_ES, SimpleES

NUM_GENERATIONS = 200

solvers_config = {
    "MPPI": {
        "solver": MPPI(population_size=N_ENVS, solution=control_init),
        "params": {"std_init": 0.25, "temperature": 2.0},
    },
    "MPPIElite": {
        "solver": MPPIElite(
            population_size=N_ENVS, solution=control_init, elite_ratio=0.1
        ),
        "params": {"std_init": 0.25, "temperature": 2.0},
    },
    "MPPICMA": {
        "solver": MPPICMA(population_size=N_ENVS, solution=control_init),
        "params": {"std_init": 0.25, "temperature": 2.0, "lr_mean": 1.0, "lr_cov": 0.1},
    },
    "CMA_ES": {
        "solver": CMA_ES(population_size=N_ENVS, solution=control_init),
        "params": {"std_init": 0.25},
    },
    "SimpleES": {
        "solver": SimpleES(population_size=N_ENVS, solution=control_init),
        "params": {"std_init": 0.25},
    },
}

results = {}
states_cartpole = {}

for name, cfg in solvers_config.items():
    print(f"\nRunning {name}...")
    es = cfg["solver"]
    params = es.default_params.replace(**cfg["params"])
    key = jax.random.key(0)
    state = es.init(key, control_init, params)
    log = []

    for _ in tqdm(range(NUM_GENERATIONS), desc=name):
        key, key_ask, key_tell = jax.random.split(key, 3)
        population, state = es.ask(key_ask, state, params)
        fitness = jnp.array(task.cost_function(np.array(population)))
        state, metrics = es.tell(key_tell, population, fitness, state, params)
        log.append(metrics)

    results[name] = {
        "generations": [m["generation_counter"] for m in log],
        "best_fitness": [m["best_fitness"] for m in log],
    }
    states_cartpole[name] = state
    print(f"  Final best fitness: {results[name]['best_fitness'][-1]:.4f}")

print("\n✓ All solvers done!")

### Convergence plot

In [ ]:
os.makedirs("Figures", exist_ok=True)

colors = {
    "MPPI": "C0",
    "MPPIElite": "C1",
    "MPPICMA": "C2",
    "CMA_ES": "C3",
    "SimpleES": "C4",
}
SOLVER_NAMES = list(solvers_config.keys())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for name, data in results.items():
    ax1.plot(
        data["generations"],
        data["best_fitness"],
        label=name,
        color=colors[name],
        linewidth=2,
        marker="o",
        markersize=2,
    )
    ax2.semilogy(
        data["generations"],
        data["best_fitness"],
        label=name,
        color=colors[name],
        linewidth=2,
        marker="o",
        markersize=2,
    )

for ax, title in zip([ax1, ax2], ["Linear scale", "Log scale"]):
    ax.set_xlabel("Generation")
    ax.set_ylabel("Best fitness")
    ax.set_title(f"CartPole — {title}")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("Figures/comparison_cartpole.png", dpi=150)
plt.show()
print("✓ Plot saved to Figures/comparison_cartpole.png")

### State and control trajectories — CartPole

In [ ]:
# Roll out each solver's best control trajectory and collect (q, v) histories
def rollout_cartpole(task, control_traj):
    """Simulate one trajectory, return q_traj (T+1,2) and v_traj (T+1,2)."""
    data = mujoco.MjData(task.mj_model)
    data.qpos[:] = task.q_init.copy()
    mujoco.mj_forward(task.mj_model, data)
    q_traj, v_traj = [], []
    for t in range(task.N_STEPS):
        q_traj.append(data.qpos.copy())
        v_traj.append(data.qvel.copy())
        for _ in range(task.N_HOLD):
            data.ctrl[:] = control_traj[t]
            mujoco.mj_step(task.mj_model, data)
    q_traj.append(data.qpos.copy())
    v_traj.append(data.qvel.copy())
    return np.array(q_traj), np.array(v_traj)


fig, axes = plt.subplots(1, 5, figsize=(20, 4))
label_cols = ["x [m]", "theta [rad]", "x_dot [m/s]", "theta_dot [rad/s]", "u"]
hlines_cfg = [
    [(task.x_max, "--"), (-task.x_max, "--")],
    [],
    [],
    [],
    [(task.u_max, "--"), (-task.u_max, "--")],
]
targets = [task.q_desired[0], task.q_desired[1], None, None, None]

t_end = None
for name, state in states_cartpole.items():
    control_traj = np.array(state.best_solution.reshape((task.N_STEPS, -1)))
    q_traj, v_traj = rollout_cartpole(task, control_traj)
    ctrl = control_traj.squeeze()

    T = q_traj.shape[0]
    time_s = np.arange(T) * task.dt * task.N_HOLD
    time_c = np.arange(len(ctrl)) * task.dt * task.N_HOLD
    t_end = time_s[-1]

    signals = [q_traj[:, 0], q_traj[:, 1], v_traj[:, 0], v_traj[:, 1], ctrl]
    times = [time_s, time_s, time_s, time_s, time_c]

    for col, (t, y) in enumerate(zip(times, signals)):
        ax = axes[col]
        ax.plot(t, y, color=colors[name], linewidth=2, label=name)
        if targets[col] is not None:
            ax.plot(t[-1], y[-1], "o", color=colors[name], markersize=8, zorder=5)

for col, ax in enumerate(axes):
    for val, ls in hlines_cfg[col]:
        ax.axhline(val, linestyle=ls, color="gray", linewidth=0.8)
    if targets[col] is not None:
        ax.plot(
            t_end,
            targets[col],
            "*",
            color="gold",
            markersize=14,
            zorder=6,
            label="target",
        )
    ax.set_title(label_cols[col])
    ax.set_xlabel("time [s]")
    ax.grid(True, alpha=0.3)

axes[0].legend(fontsize=8)
axes[1].legend(fontsize=8)

plt.suptitle("CartPole — state and control trajectories", fontsize=13)
plt.tight_layout()
plt.savefig("Figures/cartpole_trajectories.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved to Figures/cartpole_trajectories.png")

### Rollout videos — CartPole

In [ ]:
import mujoco
import imageio
from IPython.display import Video, display

print("Generating CartPole rollout videos...")

camera_id = mujoco.mj_name2id(task.mj_model, mujoco.mjtObj.mjOBJ_CAMERA, "fixed")
fps = int(round(1.0 / task.dt))

for name, state in states_cartpole.items():
    control_traj = np.array(state.best_solution.reshape((task.N_STEPS, -1)))
    data = mujoco.MjData(task.mj_model)
    data.qpos[:] = task.q_init.copy()
    mujoco.mj_forward(task.mj_model, data)
    video_path = f"Figures/cartpole_{name.lower()}.mp4"
    with mujoco.Renderer(task.mj_model, width=640, height=480) as renderer:
        with imageio.get_writer(video_path, fps=fps) as writer:
            for t in range(task.N_STEPS):
                for _ in range(task.N_HOLD):
                    data.ctrl[:] = control_traj[t]
                    mujoco.mj_step(task.mj_model, data)
                    renderer.update_scene(data, camera=camera_id)
                    writer.append_data(renderer.render())
    print(f"  ✓ {name} → {video_path}")

print("\n✓ All CartPole videos saved!")

In [ ]:
print("CartPole rollouts:\n")
for name in SOLVER_NAMES:
    video_path = f"Figures/cartpole_{name.lower()}.mp4"
    if os.path.exists(video_path):
        print(f"{name}:")
        display(Video(video_path, width=640, height=480))
    else:
        print(f"  (not found: {video_path})")

## Step 4 — Environment 2: Unitree G1 Humanoid

### System description

The [Unitree G1](https://www.unitree.com/g1/) is a full-size humanoid robot simulated in MuJoCo. We use a reduced **23-DOF** model that covers the legs, waist, and arms — the hands are fixed.

```
        [head]
    [L shoulder]──[R shoulder]
    [L elbow  ]    [R elbow  ]
    [L wrist  ]    [R wrist  ]
         │    [waist yaw]    │
    [L hip    ]──────────[R hip    ]
    [L knee   ]          [R knee   ]
    [L ankle  ]          [R ankle  ]
```

The robot is **fully actuated**: every joint has a motor. The floating base (position + orientation) is **not** actuated — it moves freely under gravity and contact.

### State space

The full state is the MuJoCo `FULLPHYSICS` state vector, which includes:

| Block | Size | Description |
|-------|------|-------------|
| `qpos` | 30 | Floating-base position (3) + quaternion (4) + 23 joint angles |
| `qvel` | 29 | Floating-base velocity (6) + 23 joint velocities |

The **initial state** is a reference standing posture loaded from `model/g1/reference_position.npy`.  
The **goal** is the same reference posture — the robot must hold a stable balanced stance.

### Action space

23 joint-position targets, one per actuator:

| Group | Joints |
|-------|--------|
| Left leg (6) | hip pitch/roll/yaw, knee, ankle pitch/roll |
| Right leg (6) | hip pitch/roll/yaw, knee, ankle pitch/roll |
| Waist (1) | yaw |
| Left arm (5) | shoulder pitch/roll/yaw, elbow, wrist roll |
| Right arm (5) | shoulder pitch/roll/yaw, elbow, wrist roll |

Each control step is held for $N_\text{hold}=5$ simulator steps (timestep $\Delta t = 0.02$ s), giving a control horizon of $T=10$ steps (~1 second total).

### Cost function

The cost has three terms, all evaluated on the **final** state plus running smoothness:

$$J = \underbrace{\sum_{t} 0.01\,\|\dot{q}_\text{acc,t}\|^2}_{\text{smoothness}} + \underbrace{e(q_T, q^*)^\top W\, e(q_T, q^*)}_{\text{terminal config error}} + \underbrace{100\,(\|p^\text{left}_T - p^{\text{left}*}\|^2 + \|p^\text{right}_T - p^{\text{right}*}\|^2)}_{\text{foot position error}}$$

where:
- $e(q, q^*)$ is the configuration error (position, quaternion-subtracted rotation, joint angles)
- $W = \text{diag}(w)$ with $w_i = 100$ for the 6 base DOFs and $w_i = 1$ for the joints — the base pose is weighted 100× more heavily
- $p^\text{left/right}$ are the 3D positions of the foot sites

### What makes this hard?

- **High-dimensional search**: the control vector is $T \times n_u = 10 \times 23 = 230$ dimensions — much larger than CartPole's 20.
- **Underactuated base**: the floating base is not directly controlled; the solver must find joint trajectories that keep the robot balanced through contact forces.
- **Short horizon**: only 10 steps to correct the posture, so the solver must act decisively from the start.

In [ ]:
from humanoid_task import Humanoid

task_h = Humanoid(N_ENVS)
control_init_h = task_h.get_warm_start()

print("Task       : Humanoid G1")
print(f"Steps      : {task_h.N_STEPS}")
print(f"Control dim: {control_init_h.shape}  (T x nu)")

In [ ]:
solvers_config_h = {
    "MPPI": {
        "solver": MPPI(population_size=N_ENVS, solution=control_init_h),
        "params": {"std_init": 0.25, "temperature": 1.0},
    },
    "MPPIElite": {
        "solver": MPPIElite(
            population_size=N_ENVS, solution=control_init_h, elite_ratio=0.1
        ),
        "params": {"std_init": 0.25, "temperature": 1.0},
    },
    "MPPICMA": {
        "solver": MPPICMA(population_size=N_ENVS, solution=control_init_h),
        "params": {"std_init": 0.25, "temperature": 0.5, "lr_mean": 1.0, "lr_cov": 0.1},
    },
    "CMA_ES": {
        "solver": CMA_ES(population_size=N_ENVS, solution=control_init_h),
        "params": {"std_init": 0.25},
    },
    "SimpleES": {
        "solver": SimpleES(population_size=N_ENVS, solution=control_init_h),
        "params": {"std_init": 0.25},
    },
}

results_h = {}
states_humanoid = {}

for name, cfg in solvers_config_h.items():
    print(f"\nRunning {name}...")
    es = cfg["solver"]
    params = es.default_params.replace(**cfg["params"])
    key = jax.random.key(0)
    state = es.init(key, control_init_h, params)
    log = []

    for _ in tqdm(range(NUM_GENERATIONS), desc=name):
        key, key_ask, key_tell = jax.random.split(key, 3)
        population, state = es.ask(key_ask, state, params)
        fitness = jnp.array(task_h.cost_function(np.array(population)))
        state, metrics = es.tell(key_tell, population, fitness, state, params)
        log.append(metrics)

    results_h[name] = {
        "generations": [m["generation_counter"] for m in log],
        "best_fitness": [m["best_fitness"] for m in log],
    }
    states_humanoid[name] = state
    print(f"  Final best fitness: {results_h[name]['best_fitness'][-1]:.4f}")

print("\n✓ All solvers done!")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for name, data in results_h.items():
    ax1.plot(
        data["generations"],
        data["best_fitness"],
        label=name,
        color=colors[name],
        linewidth=2,
        marker="o",
        markersize=2,
    )
    ax2.semilogy(
        data["generations"],
        data["best_fitness"],
        label=name,
        color=colors[name],
        linewidth=2,
        marker="o",
        markersize=2,
    )

for ax, title in zip([ax1, ax2], ["Linear scale", "Log scale"]):
    ax.set_xlabel("Generation")
    ax.set_ylabel("Best fitness")
    ax.set_title(f"Humanoid — {title}")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("Figures/comparison_humanoid.png", dpi=150)
plt.show()
print("✓ Plot saved to Figures/comparison_humanoid.png")

### Rollout videos — Humanoid

In [ ]:
print("Generating Humanoid rollout videos...")

fps_h = int(round(1.0 / task_h.dt))

for name, state in states_humanoid.items():
    control_traj = np.array(state.best_solution.reshape((task_h.N_STEPS, -1)))
    data = mujoco.MjData(task_h.mj_model)
    data.qpos[:] = task_h.q_init.copy()
    mujoco.mj_forward(task_h.mj_model, data)
    video_path = f"Figures/humanoid_{name.lower()}.mp4"
    with mujoco.Renderer(task_h.mj_model, width=640, height=480) as renderer:
        with imageio.get_writer(video_path, fps=fps_h) as writer:
            for t in range(task_h.N_STEPS):
                for _ in range(task_h.N_HOLD):
                    data.ctrl[:] = control_traj[t]
                    mujoco.mj_step(task_h.mj_model, data)
                    renderer.update_scene(data)
                    writer.append_data(renderer.render())
    print(f"  ✓ {name} → {video_path}")

print("\n✓ All Humanoid videos saved!")

In [ ]:
print("Humanoid rollouts:\n")
for name in states_humanoid:
    video_path = f"Figures/humanoid_{name.lower()}.mp4"
    if os.path.exists(video_path):
        print(f"{name}:")
        display(Video(video_path, width=640, height=480))
    else:
        print(f"  (not found: {video_path})")